# **Random Forest Model - 9 Classes**




In [ ]:
# STEP 0: Mounting Google Drive and quick check
from google.colab import drive
drive.mount('/content/drive')

!ls "/content/drive/MyDrive/CAPSTONE PROJECT"

In [ ]:
# STEP 1: Import necessary libraries
import os
import json
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Importing specific tools from scikit-learn for machine learning tasks.
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score
)
from sklearn.preprocessing import label_binarize
from sklearn.inspection import permutation_importance

In [ ]:
# STEP 2: Important Settings + Results Folder Configuration

# `SEED` is a fixed value used for reproducibility of random operations.
# Setting a seed ensures that random processes (like data splitting or model initialization) produce the same results each time.
SEED = 42
# `TARGET_COL` defines the name of the column that contains the target variable (the labels to be predicted).
TARGET_COL = "label"

# `RESULTS_DIR` specifies the path where all generated plots and CSV files will be saved.
# `os.makedirs(RESULTS_DIR, exist_ok=True)` creates the directory if it doesn't already exist,
# preventing errors if the folder is missing.
RESULTS_DIR = "/content/drive/MyDrive/CAPSTONE PROJECT/XG boost"
os.makedirs(RESULTS_DIR, exist_ok=True)

# Setting random seeds for `numpy` and `random` libraries to ensure reproducibility across different runs.
np.random.seed(SEED)
random.seed(SEED)

print("Setup done.")
print("Results will save to:", RESULTS_DIR)

In [ ]:
#STEP 3: Load the CSV + sanity check

CSV_PATH = "/content/drive/MyDrive/CAPSTONE PROJECT/9 CLASSES DATA SET/SC_Dataset_9_Classes.csv"

df = pd.read_csv(CSV_PATH)

print("Shape:", df.shape)
print("\nLabel counts:")
print(df[TARGET_COL].value_counts())

print("\nAny nulls?", df.isnull().sum().sum())

display(df.head(3))



---



# **Label Encoding**

In [ ]:
s#STEP 4: Encode labels (text to numbers)

#mapping each class name to a number
label_map = {
    "pigmented benign keratosis" : 0,
    "melanoma"                   : 1,
    "basal cell carcinoma"       : 2,
    "nevus"                      : 3,
    "squamous cell carcinoma"    : 4,
    "vascular lesion"            : 5,
    "actinic keratosis"          : 6,
    "dermatofibroma"             : 7,
    "seborrheic keratosis"       : 8
}

#apply the mapping
df[TARGET_COL] = df[TARGET_COL].map(label_map).astype(int)

#class names in order (we'll use this for plots later)
CLASS_NAMES = list(label_map.keys())

print("Label encoding done.")
print("\nEncoded label counts:")
print(df[TARGET_COL].value_counts().sort_index())
print("\nClass names:", CLASS_NAMES)

# **80/20 Split**

In [ ]:
#STEP 5: Stratified 80/20 Train/Test Split

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,          #this ensures all 9 classes are represented in both train and test
    random_state=SEED
)

print("Train size:", X_train.shape)
print("Test size :", X_test.shape)

print("\nTrain label counts:")
print(y_train.value_counts().sort_index())

print("\nTest label counts:")
print(y_test.value_counts().sort_index())

# **Feature Engineering**

In [ ]:
#STEP 6A: Channel Ratio Features
#dividing channels to capture color dominance between R, G, B

X_train["ratio_rg"] = X_train["avg_r"] / (X_train["avg_g"] + 1e-6)
X_train["ratio_rb"] = X_train["avg_r"] / (X_train["avg_b"] + 1e-6)
X_train["ratio_gb"] = X_train["avg_g"] / (X_train["avg_b"] + 1e-6)

X_test["ratio_rg"] = X_test["avg_r"] / (X_test["avg_g"] + 1e-6)
X_test["ratio_rb"] = X_test["avg_r"] / (X_test["avg_b"] + 1e-6)
X_test["ratio_gb"] = X_test["avg_g"] / (X_test["avg_b"] + 1e-6)

print("Channel ratio features added.")
print("New shape:", X_train.shape)

In [ ]:
#STEP 6B: Color Contrast Features
#subtracting channels to capture how different the colors are from each other

X_train["contrast_rg"] = X_train["avg_r"] - X_train["avg_g"]
X_train["contrast_rb"] = X_train["avg_r"] - X_train["avg_b"]
X_train["contrast_gb"] = X_train["avg_g"] - X_train["avg_b"]

X_test["contrast_rg"] = X_test["avg_r"] - X_test["avg_g"]
X_test["contrast_rb"] = X_test["avg_r"] - X_test["avg_b"]
X_test["contrast_gb"] = X_test["avg_g"] - X_test["avg_b"]

print("Color contrast features added.")
print("New shape:", X_train.shape)

In [ ]:
#STEP 6C: Color Moment Features
#overall brightness, color range, and color variation across R, G, B

X_train["brightness"]  = (X_train["avg_r"] + X_train["avg_g"] + X_train["avg_b"]) / 3
X_train["color_range"] = X_train[["avg_r","avg_g","avg_b"]].max(axis=1) - X_train[["avg_r","avg_g","avg_b"]].min(axis=1)
X_train["color_std"]   = X_train[["avg_r","avg_g","avg_b"]].std(axis=1)

X_test["brightness"]  = (X_test["avg_r"] + X_test["avg_g"] + X_test["avg_b"]) / 3
X_test["color_range"] = X_test[["avg_r","avg_g","avg_b"]].max(axis=1) - X_test[["avg_r","avg_g","avg_b"]].min(axis=1)
X_test["color_std"]   = X_test[["avg_r","avg_g","avg_b"]].std(axis=1)

print("Color moment features added.")
print("New shape:", X_train.shape)

In [ ]:
#STEP 6D: ABCD Shape Features
#asymmetry, border irregularity, and diameter estimate from existing shape columns

X_train["asymmetry_ratio"]     = X_train["min_area_rect_width"] / (X_train["min_area_rect_height"] + 1e-6)
X_train["border_irregularity"] = X_train["min_enc_circle_area"] / (X_train["min_area_rect_width"] * X_train["min_area_rect_height"] + 1e-6)
X_train["diameter_estimate"]   = (X_train["min_area_rect_width"] + X_train["min_area_rect_height"]) / 2

X_test["asymmetry_ratio"]     = X_test["min_area_rect_width"] / (X_test["min_area_rect_height"] + 1e-6)
X_test["border_irregularity"] = X_test["min_enc_circle_area"] / (X_test["min_area_rect_width"] * X_test["min_area_rect_height"] + 1e-6)
X_test["diameter_estimate"]   = (X_test["min_area_rect_width"] + X_test["min_area_rect_height"]) / 2

print("ABCD shape features added.")
print("New shape:", X_train.shape)

In [ ]:
#STEP 6E: Summary of all new engineered features

new_features = [
    "ratio_rg", "ratio_rb", "ratio_gb",
    "contrast_rg", "contrast_rb", "contrast_gb",
    "brightness", "color_range", "color_std",
    "asymmetry_ratio", "border_irregularity",
    "diameter_estimate"
]

print("Total new features added:", len(new_features))
print("Total feature count now:", X_train.shape[1])
print("\nNew features:", new_features)

# **Applying Over-Sampling to the training data**

In [ ]:
#STEP 7: Applying SMOTE on training data only

from imblearn.over_sampling import SMOTE

print("Before SMOTE:")
print(y_train.value_counts().sort_index())

smote = SMOTE(random_state=SEED)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("\nAfter SMOTE:")
print(pd.Series(y_train_sm).value_counts().sort_index())

print("\nX_train shape after SMOTE:", X_train_sm.shape)

In [ ]:
#STEP 8:Random Forest Pipeline

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("rf", RandomForestClassifier(
        n_estimators=200,
        random_state=SEED,
        n_jobs=-1
    ))
])

print("Pipeline ready.")
print(pipe)

In [ ]:
#STEP 9: Hyperparameter Tuning with RandomizedSearchCV

param_dist = {
    "rf__n_estimators":      [100, 200, 300, 400, 500], # Number of trees
    "rf__max_depth":         [None, 10, 20, 30],       # Maximum depth of the tree (None means full depth)
    "rf__min_samples_split": [2, 5, 10],               # Minimum samples required to split an internal node
    "rf__min_samples_leaf":  [1, 2, 4],                # Minimum samples required to be at a leaf node
    "rf__max_features":      ['sqrt', 'log2']          # Number of features to consider at each split
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

search = RandomizedSearchCV(
    estimator=pipe,
    param_distributions=param_dist,
    n_iter=40,
    scoring="f1_macro",
    n_jobs=-1,
    cv=cv,
    verbose=1,
    random_state=SEED
)

#training on SMOTE'd training data
search.fit(X_train_sm, y_train_sm)

print("\nBEST PARAMETERS:")
print(json.dumps(search.best_params_, indent=2))

print("\nBest CV F1 macro:", round(search.best_score_, 4))

best_model = search.best_estimator_

In [ ]:
#STEP 10: Evaluating on test set

y_pred  = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)

print("Test Accuracy:", round(accuracy_score(y_test, y_pred), 4))

print("\nClassification Report (9-class XGBoost):")
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, zero_division=0))

In [ ]:
#STEP 11: Confusion Matrix

fig, ax = plt.subplots(figsize=(12, 10))

disp = ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(y_test, y_pred, labels=list(range(9))),
    display_labels=CLASS_NAMES
)

disp.plot(
    ax=ax,
    cmap="Blues",
    values_format="d",
    colorbar=True
)

ax.set_title("Confusion Matrix — XGBoost (9-class)", fontsize=16, pad=12)
ax.set_xlabel("Predicted label", fontsize=13)
ax.set_ylabel("True label", fontsize=13)

plt.setp(ax.get_xticklabels(), rotation=35, ha="right", fontsize=10)
plt.setp(ax.get_yticklabels(), rotation=0, fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "confusion_matrix_9class.png"), dpi=150)
plt.show()

print("Saved confusion matrix.")

In [ ]:
#STEP 12: ROC-AUC Curves

y_test_bin = label_binarize(y_test, classes=list(range(9)))

auc_macro = roc_auc_score(
    y_test_bin,
    y_proba,
    average="macro",
    multi_class="ovr"
)

print("ROC-AUC macro (OvR):", round(auc_macro, 4))

plt.figure(figsize=(10, 7))

for i, cname in enumerate(CLASS_NAMES):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_proba[:, i])
    plt.plot(fpr, tpr, label=cname)

plt.plot([0, 1], [0, 1], linestyle="--", color="grey")
plt.xlabel("False Positive Rate", fontsize=13)
plt.ylabel("True Positive Rate", fontsize=13)
plt.title(f"ROC Curves — XGBoost (9-class) | macro AUC = {auc_macro:.3f}", fontsize=15)
plt.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "roc_curves_9class.png"), dpi=150)
plt.show()

print("Saved ROC curves.")

In [ ]:
#STEP 13: Feature Importance by Gain (top 30)

# Corrected: Access the RandomForestClassifier step named 'rf'
# Note: The variable name 'xgb_model' is now misleading as it holds a RandomForestClassifier.
# Also, methods like '.get_booster()' are specific to XGBoost and will not work on RandomForestClassifier.
rf_model    = best_model.named_steps["rf"]
feature_names = X_train_sm.columns.tolist()

# The following lines using .get_booster() are specific to XGBoost and will cause an AttributeError
# if xgb_model is a RandomForestClassifier.
# If you wish to continue with RandomForest, you would typically use 'feature_importances_'
# and sort them, or use a method like permutation importance (which is already in STEP 14).
# For now, this part will likely error if run with RandomForestClassifier.
# booster     = xgb_model.get_booster()
# gain_scores = booster.get_score(importance_type="gain")

# To make this cell executable with RandomForest, you would change it to something like:
imp_df = (
    pd.DataFrame({
        "feature": feature_names,
        "gain"   : rf_model.feature_importances_
    })
    .sort_values("gain", ascending=False)
    .reset_index(drop=True)
)

print("Top 20 features (gain):")
display(imp_df.head(20))

#plotting top 30
top_30 = imp_df.head(30).iloc[::-1]

plt.figure(figsize=(10, 8))
plt.barh(top_30["feature"], top_30["gain"], color="#1f77b4")
plt.xlabel("Gain", fontsize=13)
plt.ylabel("Feature", fontsize=13)
plt.title("Top 30 Feature Importances — RandomForest (9-class)", fontsize=15, pad=12)
plt.grid(axis="x", linestyle="--", alpha=0.4)
plt.xticks(fontsize=11)
plt.yticks(fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "feature_importance_gain_9class.png"), dpi=150)
plt.show()

print("Saved feature importance plot.")

In [ ]:
#STEP 14: Permutation Importance

PI_DIR = os.path.join(RESULTS_DIR, "permutation_importance")
os.makedirs(PI_DIR, exist_ok=True)

pi = permutation_importance(
    best_model,
    X_test,
    y_test,
    scoring="f1_macro",
    n_repeats=10,
    random_state=SEED,
    n_jobs=-1
)

pi_df = pd.DataFrame({
    "feature": X_test.columns,
    "pi_mean": pi.importances_mean,
    "pi_std" : pi.importances_std
}).sort_values("pi_mean", ascending=False).reset_index(drop=True)

#save full ranking
pi_df.to_csv(os.path.join(PI_DIR, "permutation_importance_full.csv"), index=False)

print("Top 10 features by permutation importance:")
display(pi_df.head(10))

#plot top 30
top_30 = pi_df.head(30).iloc[::-1]

plt.figure(figsize=(10, 8))
plt.barh(top_30["feature"], top_30["pi_mean"], xerr=top_30["pi_std"])
plt.xlabel("Mean decrease in F1 macro after shuffling", fontsize=12)
plt.title("Top 30 Permutation Importances — XGBoost (9-class)", fontsize=15)
plt.grid(axis="x", linestyle="--", alpha=0.35)
plt.tight_layout()
plt.savefig(os.path.join(PI_DIR, "permutation_importance_top30.png"), dpi=150)
plt.show()

print("Saved permutation importance plot.")

### SHAP Waterfall Plot for a Single Prediction

To understand how individual features contribute to a specific prediction, we can use a SHAP waterfall plot. This plot visualizes the impact of each feature, pushing the prediction from the base value to the model's output for that instance.

In [ ]:
import shap
import random

# Choose a random instance from the test set
instance_idx = random.randint(0, len(X_test) - 1)
X_test_instance = X_test.iloc[[instance_idx]]
y_test_instance = y_test.iloc[instance_idx]

print(f"Displaying SHAP waterfall plot for instance index: {instance_idx}")
print(f"True label for this instance: {CLASS_NAMES[y_test_instance]}")

# Re-initialize explainer with the best model (RandomForestClassifier)
shap_model = best_model.named_steps["rf"]
explainer = shap.TreeExplainer(shap_model)

# Get SHAP values for the selected instance
# We need to specify the output class if shap_vals is a list of arrays (multi-output)
# Let's visualize for the predicted class or a specific class (e.g., class 1: melanoma)

# Predict the class for the instance to show SHAP for the predicted class
predicted_class_idx = best_model.predict(X_test_instance)[0]
predicted_class_name = CLASS_NAMES[predicted_class_idx]

print(f"Predicted label for this instance: {predicted_class_name}")

# SHAP values for the predicted class
# If shap_list is a list of (n_samples, n_features) arrays (like in the notebook output)
# Ensure shap_list is available. If not, recompute it as in STEP 15.
if 'shap_list' not in locals():
    try:
        shap_exp  = explainer(X_test)
        shap_vals = shap_exp.values
    except:
        shap_vals = explainer.shap_values(X_test)

    #normalize to (n_samples, n_features, n_classes)
    if isinstance(shap_vals, np.ndarray) and shap_vals.ndim == 3:
        shap_list = [shap_vals[:, :, i] for i in range(9)]
    elif isinstance(shap_vals, list):
        shap_list = shap_vals


if isinstance(shap_list, list):
    shap_values_for_instance = shap_list[predicted_class_idx][instance_idx]
else:
    # If shap_vals is a (n_samples, n_features, n_classes) array
    shap_values_for_instance = shap_vals[instance_idx, :, predicted_class_idx]

# Create a SHAP Explanation object for the specific instance and predicted class
explanation = shap.Explanation(
    values=shap_values_for_instance,
    base_values=explainer.expected_value[predicted_class_idx], # Expected value for the predicted class
    data=X_test_instance.values[0], # Feature values for the instance
    feature_names=X_test.columns.tolist()
)

# Plot the waterfall plot
plt.figure(figsize=(10, 6))
shap.waterfall_plot(explanation, max_display=20, show=False)
plt.title(f"SHAP Waterfall Plot for Instance {instance_idx} (Predicted: {predicted_class_name})")
plt.tight_layout()
plt.show()

print("SHAP waterfall plot generated.")

In [ ]:
#STEP 15: SHAP Analysis

import shap

SHAP_DIR = os.path.join(RESULTS_DIR, "shap")
os.makedirs(SHAP_DIR, exist_ok=True)

shap_model = best_model.named_steps["rf"] # Corrected: Changed 'xgb' to 'rf'

explainer = shap.TreeExplainer(shap_model)

try:
    shap_exp  = explainer(X_test)
    shap_vals = shap_exp.values
except:
    shap_vals = explainer.shap_values(X_test)

#normalize to (n_samples, n_features, n_classes)
if isinstance(shap_vals, np.ndarray) and shap_vals.ndim == 3:
    shap_list = [shap_vals[:, :, i] for i in range(9)]
elif isinstance(shap_vals, list):
    shap_list = shap_vals

print("SHAP done.")
print("Shape per class:", shap_list[0].shape)

#summary plot per class
for i, cname in enumerate(CLASS_NAMES):
    plt.figure()
    shap.summary_plot(shap_list[i], X_test, show=False)
    plt.title(f"SHAP Summary — {cname}")
    out_path = os.path.join(SHAP_DIR, f"shap_summary_{i}_{cname.replace(' ','_')}.png")
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved:", out_path)

#global importance
mean_abs_shap = np.mean([np.abs(sv) for sv in shap_list], axis=(0, 1))

shap_global_df = pd.DataFrame({
    "feature"      : X_test.columns,
    "mean_abs_shap": mean_abs_shap
}).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

shap_global_df.to_csv(os.path.join(SHAP_DIR, "shap_global_importance.csv"), index=False)

print("\nTop 10 global SHAP features:")
display(shap_global_df.head(10))

#global bar plot
top_30 = shap_global_df.head(30).iloc[::-1]

plt.figure(figsize=(10, 8))
plt.barh(top_30["feature"], top_30["mean_abs_shap"])
plt.xlabel("Mean |SHAP value|", fontsize=13)
plt.title("Top 30 Global SHAP Importances — RandomForest (9-class)", fontsize=15) # Updated title
plt.tight_layout()
plt.savefig(os.path.join(SHAP_DIR, "shap_global_top30.png"), dpi=150)
plt.show()

print("Saved global SHAP plot.")

### SHAP Force Plot for a Single Prediction

A SHAP force plot is another interactive visualization that shows how each feature contributes to pushing the model's output from the base value (average prediction) to the prediction for a specific instance. Features pushing the prediction higher are shown in red, and those pushing it lower are in blue.

In [ ]:
import shap
import random

# Choose a random instance from the test set for the force plot
instance_idx_fp = random.randint(0, len(X_test) - 1)
X_test_instance_fp = X_test.iloc[[instance_idx_fp]]
y_test_instance_fp = y_test.iloc[instance_idx_fp]

print(f"Displaying SHAP force plot for instance index: {instance_idx_fp}")
print(f"True label for this instance: {CLASS_NAMES[y_test_instance_fp]}")

# Reusing the explainer from previous SHAP analysis
# shap_model = best_model.named_steps["rf"]
# explainer = shap.TreeExplainer(shap_model)

# Predict the class for the instance to show SHAP for the predicted class
predicted_class_idx_fp = best_model.predict(X_test_instance_fp)[0]
predicted_class_name_fp = CLASS_NAMES[predicted_class_idx_fp]

print(f"Predicted label for this instance: {predicted_class_name_fp}")

# Get SHAP values for the selected instance for the predicted class
# This assumes shap_list is already available from previous SHAP analysis (STEP 15)
if isinstance(shap_list, list):
    shap_values_for_instance_fp = shap_list[predicted_class_idx_fp][instance_idx_fp]
else:
    # If shap_vals is a (n_samples, n_features, n_classes) array
    shap_values_for_instance_fp = shap_vals[instance_idx_fp, :, predicted_class_idx_fp]

# Get the base value (expected value) for the predicted class
base_value_fp = explainer.expected_value[predicted_class_idx_fp]

# Plot the force plot
# Note: shap.force_plot() is an interactive plot, so it might not display directly
# in some environments without specific renderer setup. It's often viewed in notebooks.
shap.initjs() # Initialize JavaScript for interactive plots

print(f"Generating force plot for predicted class: {predicted_class_name_fp}")
shap.force_plot(
    base_value_fp,
    shap_values_for_instance_fp,
    X_test_instance_fp.values[0],
    feature_names=X_test.columns.tolist()
)

print("SHAP force plot generated.")

In [ ]:
#STEP 16: Save Predictions + Error Analysis

results_df = X_test.copy()
results_df["true_label"] = y_test.values
results_df["pred_label"] = y_pred

#add probabilities for each class
prob_cols = [f"prob_{cname.replace(' ','_')}" for cname in CLASS_NAMES]
results_df[prob_cols] = y_proba

#save
save_path = os.path.join(RESULTS_DIR, "xgb_9class_predictions.csv")
results_df.to_csv(save_path, index=False)
print("Saved predictions to:", save_path)

#correct vs incorrect
results_df["correct"] = results_df["true_label"] == results_df["pred_label"]

print("\nCorrect vs incorrect:")
print(results_df["correct"].value_counts())

#where is the model getting confused
confusion_breakdown = (
    results_df
    .loc[~results_df["correct"], ["true_label", "pred_label"]]
    .value_counts()
    .reset_index(name="count")
    .head(15)
)

print("\nTop 15 misclassifications:")
display(confusion_breakdown)